# Playground Series S6E8 (Predicting Smartphone Addiction) / 「S6E8 the score I would not pick」解説付き写し

- **コンペ**: [Predicting Smartphone Addiction — Playground Series S6E8](https://www.kaggle.com/competitions/playground-series-s6e8)
- **元notebook**: [S6E8 the score I would not pick](https://www.kaggle.com/code/szymonkapiski/s6e8-the-score-i-would-not-pick) (Szymon Kłapiński)
- **Best Score**: 0.97119 (V7) — 本日時点で公開LB検索結果の最高値 / 22 votes / 実行20分36秒
- **ライセンス**: Apache 2.0

> これは **学習目的の解説付き写し** です。原著者のコードは一字一句そのまま保持し、コードセルの直前に日本語の解説Markdownセルを挿入しています(原著者自身の英語Markdownセルもそのまま残しています)。出力は含めていません。

## なぜこのnotebookを選んだか

S6E8 はこれまで20本近く扱ってきたが、公開LB上位帯はどれも「他人の提出CSVをランク平均しただけ」の薄い内容になりがちだった。本notebookは**最高スコアでありながら、著者自身が「このスコアは最終提出に選ばない」と宣言し、その理由を検証可能な形で提示している**という点で異質。スタッキングの技術に加えて、**プロヴェナンス(出所)監査**という、実務のモデルリスク管理に直結する視点が学べる。

## 手法の概要

1. 公開されている多数の **OOF ライブラリ**(他のKagglerが公開した out-of-fold 予測配列)を `/kaggle/input` からグロブで収集し、4種類のファイルレイアウトを吸収して読み込む
2. 3つのスクリーニング(内容ハッシュによる重複排除 / solo AUC < 0.90 の除外 / OOF と test の分布が乖離しているもの[KS統計量 > 0.05]の除外)
3. 生き残ったメンバーを **rank-gauss 変換**してメタ特徴にし、**L2正則化ロジスティック回帰(C=0.03)** でスタック。同じ5分割で **nested(入れ子)** に評価
4. 著者自身の「最も弱い50モデル」(**生成モデル系**、勾配ブースティングでもニューラルネットでもない)を追加し、寄与を測定
5. プール各メンバーの**分割スキームが検証可能かどうか**を階層に分けて開示し、検証不能なメンバーがメタモデルの重みをどう歪めるかを論じる

## 評価指標

- **タスク**: 表形式データから `addicted_label`(スマホ依存かどうか)を予測する**二値分類**。
- **指標**: **ROC-AUC**。予測値の「順序」だけを見る指標で、絶対値のキャリブレーションは問われない(最終セルで `rank01` を掛けているのはそのため — 順序が変わらないので AUC は不変)。
- **なぜAUCか**: 陽性率が偏った二値分類で、閾値をどこに置くかが用途次第のときに使う。Accuracy は閾値と陽性率に強く依存するが、AUC は「無作為に選んだ陽性の予測値が、無作為に選んだ陰性より高い確率」に等しく、**閾値に依存しない順位付け能力**を測る。
- **この手法が指標をどう最適化しているか**: AUC が順序しか見ないため、(a) メンバーを **rank-gauss** に変換して尺度差を消す、(b) 最後に **rank01** で [0,1] に均す、といった順序保存変換が自由に使える。スタックはロジスティック回帰の `decision_function`(確率化前のスコア)をそのまま使っており、これも順序さえ合えば十分という判断。
- **著者の核心的な主張**: このコンペの上位クラスタは **0.9711〜0.9713 の幅に密集**しており、**同一に近い提出を繰り返しただけで動く範囲**に収まっている。つまり**公開LBはこの帯域の順位を決める分解能を持っていない**。だから「公開LBが最高だから選ぶ」は意思決定として成立しない、というのが本notebookのタイトルの意味。

# The score I would not pick as my final submission

This notebook does two things.

First it reproduces a public stacking pipeline over the public out-of-fold libraries for
this competition, then adds **50 of my own weakest models** to it and measures what they
are worth. The 50 arrays are published alongside this notebook as a new dataset.

Second, and this is the part I think is more useful, it lays out **which members of that
pool I can verify and which I cannot**, and explains why the highest public score I have
produced is not the submission I intend to select at the deadline.

Those two things are connected. The pool that scores highest contains arrays whose fold
scheme I cannot verify, and one whose scheme I can verify and which does not match. I am
reproducing the pool faithfully rather than quietly cleaning it, because a notebook about
provenance that silently drops its own inconvenient members is not about provenance.

## About the score next to my name

If you look at the public leaderboard for this competition you will currently find me at
**0.97130**. Here is where it came from and what I am going to do with it.

It came from exactly the experiment in this notebook, run to its end: this same public pool,
plus **all** of my unpublished out-of-fold library rather than the 50 arrays published here.
It is the highest score I have produced.

I am not going to select it. Not because I doubt the arithmetic, but because **not all of
the out-of-fold arrays in this pool are fully verifiable**, and a blend whose weights rest on
arrays I cannot account for is a worse bet on the private split than a slightly lower score
built on arrays I can. The last section is the full argument.

One of those arrays is my own responsibility, so I will put it first. My published OOF
library includes `pub_evg`, which uses a 10-fold out-of-fold protocol rather than the 5-fold
one the rest of the pool assumes. I did not train it, I adopted it, and I did not check it
before including it. My library description previously stated that every model in it used
the 5-fold split with seed 42. That was wrong, and this is the first place I have said so in
public.

This notebook is the honest half of that: here is the pool, here is what it contains, here
is what my own weakest models add to it, and here is why the number it produces is not the
number I am betting on.

## What is new here

The 50 arrays. They are the weakest models in my library, they are all mine, and they are
all out-of-fold under the same 5-fold split this community has converged on. Solo they are
poor. Added to a strong public pool they still move it, and the reason is not subtle once you
look at what they are.

Worth separating from everything above: What I would not stake a final submission on is the pool, not the dataset I am adding to it. If you only want the clean piece, take the dataset. Although, I do not publish training code of each model - you can only partially trust me to be fair.

## What is borrowed

The stacking pipeline and the pool assembly are **adarsh1077's**, from
[S6E8 Diversity Beats Strength](https://www.kaggle.com/code/adarsh1077/s6e8-diversity-beats-strength).
The idea of attaching many public OOF libraries and stacking them is his, the choice of
rank-gauss meta features is his, and the list of datasets attached to this notebook is the
list he assembled. I reimplemented the pipeline on my own folds rather than copying his
code, and I have kept his hyperparameters. Every source is credited in the next section.

## Credits

Up front rather than at the bottom, because most of what this notebook runs on is other
people's work. Every dataset attached here, and every author whose models are in the pool.

**Pipeline and pool assembly**
[adarsh1077, S6E8 Diversity Beats Strength](https://www.kaggle.com/code/adarsh1077/s6e8-diversity-beats-strength).
The stacking approach, the rank-gauss meta features, the `C=0.03` setting, the 0.90
quarantine and the list of attached datasets are all from that notebook. I reimplemented the
pipeline rather than copying the code, and no code from it is included here.

**OOF libraries in the pool**

| author | dataset |
|---|---|
| adarsh1077 | [s6e8-adarsh-oof-library](https://www.kaggle.com/datasets/adarsh1077/s6e8-adarsh-oof-library) |
| beicicc | [xgb-identity-digit](https://www.kaggle.com/datasets/beicicc/s6e8-fixed1500-xgb-identity-digit-artifacts), [xgb-screen-relation](https://www.kaggle.com/datasets/beicicc/s6e8-fixed1500-xgb-screen-relation-artifacts), [realmlp-two-seed](https://www.kaggle.com/datasets/beicicc/s6e8-fixed4-realmlp-two-seed-artifacts), [identity-digit-lightgbm](https://www.kaggle.com/datasets/beicicc/s6e8-fixed900-identity-digit-lightgbm-artifacts), [exact-value-catboost](https://www.kaggle.com/datasets/beicicc/s6e8-fixed-schedule-exact-value-catboost-artifacts), [lookup-transformer](https://www.kaggle.com/datasets/beicicc/s6e8-fixed-schedule-lookup-transformer-artifacts), [structural-lgbm](https://www.kaggle.com/datasets/beicicc/s6e8-fixed900-structural-lgbm-artifacts), [second-seed-lookup](https://www.kaggle.com/datasets/beicicc/s6e8-second-seed-fixed-schedule-lookup-artifacts), [sixmember-crossfit-logitlr](https://www.kaggle.com/datasets/beicicc/s6e8-sixmember-crossfit-logitlr-artifacts) |
| boltuzamaki | [s6e8-oof-prediction-library](https://www.kaggle.com/datasets/boltuzamaki/s6e8-oof-prediction-library), and his [stacking writeup](https://www.kaggle.com/code/boltuzamaki/stacking-47-models-public-lb-0-97073), which is where the fold protocol quoted above comes from |
| dariushafshar | [s6e8-golem-oof-library](https://www.kaggle.com/datasets/dariushafshar/s6e8-golem-oof-library), [s6e8-measured-findings-pack](https://www.kaggle.com/datasets/dariushafshar/s6e8-measured-findings-pack) |
| mohankrishnathalla | [lgb-dart-oof](https://www.kaggle.com/datasets/mohankrishnathalla/s6e8-lgb-dart-oof), [xgb-oof](https://www.kaggle.com/datasets/mohankrishnathalla/s6e8-xgb-oof), [cat-mlp-oof](https://www.kaggle.com/datasets/mohankrishnathalla/s6e8-cat-mlp-oof) |
| najiama | [oof-submission-csv](https://www.kaggle.com/datasets/najiama/predicting-smartphone-addiction-oof-submission-csv) |
| raykkretzschmar | [s6e8-fm-lattice-blend-members](https://www.kaggle.com/datasets/raykkretzschmar/s6e8-fm-lattice-blend-members) |
| szymonkapiski (me) | [s6e8-oof-library-47-models](https://www.kaggle.com/datasets/szymonkapiski/s6e8-oof-library-47-models) |

**Mechanisms behind models in my own library that appear in the pool**, credited where I
reimplemented someone else's published idea on my own folds:

- [FunnyBishop, value lookup CatBoost](https://www.kaggle.com/code/funnybishop/s6e8-value-lookup-catboost)
- [tamerlanomralinov, Lookup Transformer](https://www.kaggle.com/code/tamerlanomralinov/s6e8-lookup-transformer-insights-lb-0-97041)
- [byerscrip, XGBoost with constrained imputation geometry](https://www.kaggle.com/code/byerscrip/s6e8-xgboost-public-score-0-96983)
- [raykkretzschmar, band local factorization machine](https://www.kaggle.com/datasets/raykkretzschmar/s6e8-fm-lattice-blend-members)
- [boltuzamaki, stacking 47 models](https://www.kaggle.com/code/boltuzamaki/stacking-47-models-public-lb-0-97073)

**The 50 arrays**: mine, CC0, published as a dataset alongside this notebook.

## Setup

Nothing here is hardcoded to a mount path. The competition frame and every attached
dataset are discovered by globbing `/kaggle/input`.

### コードの読み方: データの発見をハードコードしない

**何をしているか**: `glob` で `/kaggle/input` 配下を再帰的に探し、`train.csv` / `test.csv` を見つける。パス中に `oof` を含むものは除外している(添付されたOOFライブラリにも同名ファイルがあり得るため)。

**なぜそうするのか**: 添付データセットの数や名前は fork するたびに変わる。マウントパスを直書きすると他人がフォークした瞬間に壊れるので、**構造(ファイル名の規約)で探す**ほうが頑健。`oof` を含むパスを弾く一行が、まさにその副作用への対処。

In [ ]:
import glob
import hashlib
import os
import warnings

import numpy as np
import pandas as pd
from scipy.stats import norm, rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

TRAIN = glob.glob("/kaggle/input/**/train.csv", recursive=True)
TEST = glob.glob("/kaggle/input/**/test.csv", recursive=True)
TRAIN = [p for p in TRAIN if "oof" not in p.lower()][0]
TEST = [p for p in TEST if "oof" not in p.lower()][0]

tr = pd.read_csv(TRAIN)
te = pd.read_csv(TEST)
y = tr["addicted_label"].to_numpy(np.int8)
n, n_te = len(tr), len(te)
print(f"train {tr.shape}   test {te.shape}   positive rate {y.mean():.4f}")

## The split

The stack below is fitted on one split:

    StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

**Most of the arrays in the pool were built on that split. Some were not.** That is the
honest version of a sentence I have seen written as "all" more than once, including by me.

It matters because an out-of-fold array is comparable to another one to the extent that both
came from the same partition. Where they did not, the array is not useless and its test
predictions are not compromised. What happens instead is that its out-of-fold values look
better than they should, so the meta model gives it more weight than it has earned. There is
a section on that below.

### 分割の固定 — すべての比較の基準線

**何をしているか**: `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` を1回だけ作り、`FOLDS` として以降すべてで使い回す。各foldの陽性率も印字して確認している。

**なぜそうするのか**: スタッキングでは、**すべてのメンバーのOOF配列が同一の分割から作られていること**が前提になる。分割が違うメンバーは「自分の検証行を訓練時に見てしまっている」ため、OOFスコアが実力より良く見える。この前提が本notebookの中心テーマなので、ここで分割を固定して明示することが議論の出発点になる。

> **用語**: *StratifiedKFold(層化K分割)* = 各foldの陽性率が全体と揃うように分割する方法。不均衡データで分散を抑えるための基本作法。

In [ ]:
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
FOLDS = list(SKF.split(np.zeros(n), y))
for i, (itr, iva) in enumerate(FOLDS):
    print(f"fold {i}: train {len(itr):,}  valid {len(iva):,}  pos rate {y[iva].mean():.5f}")

## Loading the pool

The attached datasets do not share a file convention. There are four layouts among them,
and the loader below handles all four:

1. `.npy` pairs named `oof_X.npy` and `test_X.npy`.
2. `.npy` pairs named `X_oof.npy` and `X_test.npy`.
3. `.parquet` files with an `id` column and one column per member.
4. `.csv` blend pairs, which is how najiama publishes.

**Credit, and this is not a small one:** the four layouts, the pairing rules and the
screening below are all from adarsh1077's notebook linked at the top. I read his loader and
wrote my own against the same conventions. One detail I took directly because it is the kind
of thing you only find by losing members to it: when deriving a test filename from an OOF
filename, replace `oof` with `test` **in the basename only**. A whole path replace turns the
folder `s6e8-oof-prediction-library` into `s6e8-test-prediction-library`, which silently
drops every member in it.

Three screens, all his:

1. **Deduplicate by content hash.** Several of these libraries mirror each other, so the same
   array can arrive twice under two names.
2. **Drop anything below 0.90 solo AUC.**
3. **Drop anything whose OOF and test rank distributions differ**, by a two sample KS
   statistic above 0.05 on a 40,000 row subsample. An array whose test predictions do not
   look like its OOF predictions is not usable as a stacking member regardless of its AUC.

### プールの読み込み — 4種類のファイルレイアウトを吸収する

**何をしているか**: 公開されたOOFライブラリは著者ごとに保存形式がバラバラ(`oof_X.npy`/`test_X.npy`、`X_oof.npy`/`X_test.npy`、id列付き parquet、najiama 形式の CSV ペア)。この関数群はその4種すべてを同じ `POOL_O`(OOF)/`POOL_T`(test)辞書に取り込み、あわせて出所(`PROV`)も記録する。

**なぜそうするのか**: 外部データを統合する処理では、**フォーマットの多様性そのものがバグの温床**になる。原著者が強調する落とし穴が象徴的 — OOFファイル名から test ファイル名を導くとき、`oof`→`test` の置換は **basename だけ**に適用しなければならない。パス全体で置換すると `s6e8-oof-prediction-library` というフォルダ名まで書き換わり、**そのフォルダのメンバーが1件も読み込まれないまま、エラーも出さずに黙って消える**。

こうした「静かに減る」バグは、件数を印字しない限り気づけない。だから次のセルで必ず件数を出す。

In [ ]:
from scipy.stats import ks_2samp

R01 = lambda v: (rankdata(v, method="average") - 0.5) / len(v)
tr_id, te_id = tr["id"].values, te["id"].values
POOL_O, POOL_T, PROV = {}, {}, {}


def add(name, o, t, src):
    o = np.asarray(o, np.float64).ravel()
    t = np.asarray(t, np.float64).ravel()
    if o.shape != (n,) or t.shape != (n_te,) or name in POOL_O:
        return
    POOL_O[name], POOL_T[name] = o, t
    parts = src.replace(chr(92), "/").split("/")
    PROV[name] = parts[3] if len(parts) > 3 else ""


# layout 1 and 2: .npy pairs, in both namings
for p in glob.glob("/kaggle/input/**/*.npy", recursive=True):
    b, d = os.path.basename(p)[:-4], os.path.dirname(p)
    if b.startswith("oof_"):
        tp, key = os.path.join(d, f"test_{b[4:]}.npy"), b[4:]
    elif b.endswith("_oof"):
        tp, key = os.path.join(d, f"{b[:-4]}_test.npy"), b[:-4]
    else:
        continue
    if os.path.exists(tp):
        add(key, np.load(p), np.load(tp), p)

# layout 3: parquet with an id column, one column per member.
# oof -> test in the BASENAME only. A whole path replace mangles the containing folder.
for p in glob.glob("/kaggle/input/**/*oof*.parquet", recursive=True):
    tp = os.path.join(os.path.dirname(p), os.path.basename(p).replace("oof", "test"))
    if not os.path.exists(tp):
        continue
    do, dt = pd.read_parquet(p), pd.read_parquet(tp)
    if "id" not in do.columns:
        continue
    do = do.set_index("id").reindex(tr_id)
    dt = dt.set_index("id").reindex(te_id)
    for c in do.columns:
        if c in dt.columns:
            add(c, do[c].to_numpy(np.float64), dt[c].to_numpy(np.float64), p)

# layout 4: najiama publishes blends as csv pairs
for p in glob.glob("/kaggle/input/**/*_blend_oof_predictions.csv", recursive=True):
    k = os.path.basename(p).split("_")[0]
    sp = os.path.join(os.path.dirname(p), f"{k}_blend_submission.csv")
    if not os.path.exists(sp):
        continue
    do, dt = pd.read_csv(p), pd.read_csv(sp)
    oc = [c for c in do.columns if c.lower() != "id"]
    tc = [c for c in dt.columns if c.lower() != "id"]
    if not oc or not tc:
        continue
    if "id" in do.columns:
        do = do.set_index("id").reindex(tr_id)
    if "id" in dt.columns:
        dt = dt.set_index("id").reindex(te_id)
    add(f"naji_blend{k}", do[oc[0]].to_numpy(np.float64),
        dt[tc[0]].to_numpy(np.float64), p)

print(f"collected {len(POOL_O)} members before screening")

### 3つのスクリーニング — 何を、なぜ捨てるか

**何をしているか**:
1. **内容ハッシュによる完全重複の除去**: 複数のライブラリが互いをミラーしているため、同一配列が別名で二重に入る。
2. **solo AUC < 0.90 の除外**: 明らかに壊れた/弱すぎる配列を落とす。
3. **KS統計量 > 0.05 の除外**: OOF とテスト予測の分布を Kolmogorov–Smirnov 検定で比較し、**大きくズレているものを落とす**。

**なぜそうするのか**:
- 重複はメタモデルの重みを不当に膨らませる(同じ意見を2票入れるのと同じ)。
- 分布の乖離は、**そのメンバーの test 予測が OOF と同じ手続きで作られていない**サインで、スタックの前提が壊れていることを示す。AUCのような性能指標では捕まらない異常を、分布比較なら捕まえられる。
- `corrector = "perp" in nm` で一部を意図的に免除しているのは、「わざと歪んだ補正用配列」が存在するため。**例外は必ず理由付きで明示する**のが良い作法。

> **用語**: *KS統計量 (Kolmogorov–Smirnov)* = 2つの経験分布の累積分布関数の最大差。分布が同じかどうかをノンパラメトリックに測る。

In [ ]:
# The three screens, then the surviving pool.
seen, dups = {}, []
for nm in sorted(POOL_O):
    h = hashlib.md5(np.ascontiguousarray(POOL_O[nm]).tobytes()).hexdigest()
    if h in seen:
        dups.append((nm, seen[h]))
        del POOL_O[nm], POOL_T[nm]
    else:
        seen[h] = nm
print(f"exact duplicates removed: {len(dups)}")

rng = np.random.default_rng(0)
ia, ib = rng.choice(n, 40000, False), rng.choice(n_te, 40000, False)
keep, dropped = [], []
for nm in sorted(POOL_O):
    o, t = POOL_O[nm], POOL_T[nm]
    au = roc_auc_score(y, o)
    corrector = "perp" in nm
    if not (np.isfinite(o).all() and np.isfinite(t).all()):
        dropped.append((nm, "nonfinite", au))
    elif au < 0.90 and not corrector:
        dropped.append((nm, "auc<0.90", au))
    else:
        ks = ks_2samp(R01(o)[ia], R01(t)[ib]).statistic
        if ks > 0.05 and not corrector:
            dropped.append((nm, f"ks={ks:.3f}", au))
        else:
            keep.append(nm)
print(f"kept {len(keep)}, dropped {len(dropped)}")
for nm, why, au in dropped:
    print(f"   drop {nm:<34s} {why:<12s} auc={au:.6f}")

POOL_META = pd.DataFrame([{"member": nm, "solo_auc": roc_auc_score(y, POOL_O[nm]),
                           "source": PROV.get(nm, "")} for nm in keep])
POOL_O = {nm: POOL_O[nm] for nm in keep}
POOL_T = {nm: POOL_T[nm] for nm in keep}
print(f"\npool members: {len(POOL_O)}")
print(f"solo AUC range: {POOL_META.solo_auc.min():.6f} to {POOL_META.solo_auc.max():.6f}")
POOL_META.groupby("source").agg(members=("member", "size"),
                                best=("solo_auc", "max")).sort_values(
    "members", ascending=False)

### Which of those members are level 2

Before anything is fitted, separate the base models from the stacks. Everything after this
point uses the base models only, for two reasons: a blend of the pool does not belong inside
the pool's own consensus, and it does not belong in a submission from a notebook arguing
that level 2 members do not belong in a final selection.

### レベル2メンバーの分離 — 「プールのブレンド」はプールの一員ではない

**何をしているか**: プール内のうち、それ自体が「プールのブレンド」であるもの(najiama の `*_blend`、および自分のライブラリ経由で紛れ込んだ `pub_ravi` というスタック)を **L2(レベル2)** として分離し、以降は**ベースモデルだけ**を使う。最強のベースモデルの solo AUC も印字し、「この線より上にあるものはデータのモデルではなくプールのブレンドだ」と明言する。

**なぜそうするのか**: スタックの入力にスタックを入れると、そのメンバーは**隣にいる他メンバーの重み付け直し**になり、メタモデルの重み推定が循環する。判別は「名前で断定」ではなく**規約から計算**している点も重要(`startswith("naji_blend")`)。手で書いたリストは必ず腐る。

In [ ]:
# Which pool members are level 2, computed rather than asserted. najiama publishes blends
# under *_blend_oof_predictions.csv, which the loader names naji_blend*, and pub_ravi is a
# stack that reached this pool through my own published library.
L2 = sorted([k for k in POOL_O if k.startswith("naji_blend") or k == "pub_ravi"])
BASE_ONLY = [k for k in sorted(POOL_O) if k not in L2]
print(f"pool loaded          {len(POOL_O)} members")
print(f"level 2 members      {len(L2)}")
print(f"base models only     {len(BASE_ONLY)}")
print()
for k in L2:
    print(f"   {k:<28s} solo AUC {roc_auc_score(y, POOL_O[k]):.6f}")
_best_base = max(roc_auc_score(y, POOL_O[k]) for k in BASE_ONLY)
print(f"\nstrongest BASE model in the pool: {_best_base:.6f}")
print("anything above that line is a blend of the pool, not a model of the data.")

## My 50 weakest models

These come from the dataset published with this notebook. Every one of them is mine, and
every one is out-of-fold under the same `StratifiedKFold(5, shuffle=True, random_state=42)`
on `train.csv` in its original row order.

I am publishing the predictions and the fold scheme, which is what you need in order to use
them. The columns are named `m01` to `m50` in ascending solo AUC.

I am holding back the architectures and the hyperparameters until the competition closes,
but not the general direction, because the direction is the interesting part: **these are
neither gradient boosted trees nor neural networks.** There is more on what that means
further down.

One thing they are not is selected. These are the 50 lowest solo out-of-fold AUCs in my
library. I did not probe the leaderboard to choose them and I did not tune the cut.

### 自作の「最も弱い50モデル」を読み込む

**何をしているか**: 著者が本notebookと同時に公開したデータセットから、50本の OOF/test 配列とメタ情報を読み込む。特徴的なのは **shipped の AUC を信用せず、その場で再計算している**こと。

**なぜそうするのか**: 「データセットに書いてある数字」と「実際の配列から計算した数字」がズレる事故は珍しくない(生成後に配列だけ差し替えた、行順が変わった等)。**検算できるものは検算する**。

また `assert` で `MINE_O.shape == (n, len(MINE_META))` を確認しているのは、行数がずれたまま気づかず学習が走るのを防ぐため。形状アサーションはコストほぼゼロで最も効く防御。

In [ ]:
_m = [f for f in glob.glob("/kaggle/input/**/members.csv", recursive=True)
      if os.path.exists(os.path.join(os.path.dirname(f), "oof.npy"))]
assert _m, ("the s6e8-50-weakest-oof-models dataset is not attached; mounts seen: "
            + str(sorted(os.path.basename(d) for d in glob.glob("/kaggle/input/*"))))
MINE_DIR = os.path.dirname(_m[0])
print(f"my dataset mounted at {MINE_DIR}")

MINE_O = np.load(os.path.join(MINE_DIR, "oof.npy"))
MINE_T = np.load(os.path.join(MINE_DIR, "test.npy"))
MINE_META = pd.read_csv(os.path.join(MINE_DIR, "members.csv"))
assert MINE_O.shape == (n, len(MINE_META)) and MINE_T.shape == (n_te, len(MINE_META))

# recompute the solo AUCs rather than trusting the shipped file
MINE_META["recomputed_auc"] = [roc_auc_score(y, MINE_O[:, j]) for j in range(MINE_O.shape[1])]
print(f"my arrays: {MINE_O.shape[1]}")
print(f"solo AUC {MINE_META.recomputed_auc.min():.6f} to {MINE_META.recomputed_auc.max():.6f}")
# Cite the strongest BASE model, not the pool maximum. The pool maximum is a naji blend,
# and a blend of the pool is not a model of the data.
_bb = max(roc_auc_score(y, POOL_O[k]) for k in BASE_ONLY)
print(f"strongest base model in the pool is {_bb:.6f}, so every array of mine here is below "
      f"every model in the pool")
MINE_META.head(10)

## The pipeline

This is adarsh1077's, reimplemented. Three steps:

1. Map each member's predictions to a rank-gauss meta feature, so members on different
   output scales are comparable.
2. Fit an L2 logistic regression over those features, `C=0.03`.
3. Do it **nested** on the same 5 folds, so the reported number is out-of-fold at the meta
   level as well as the member level. A stack scored in-sample reads high and means nothing.

Then one refit on all rows produces the test prediction.

### パイプライン本体 — rank-gauss + L2ロジスティック回帰 + nested評価

**何をしているか**:
1. `rank01`: 値を順位に変換して [0,1] に写す
2. `gauss`: それを正規分布の分位点に写す(**rank-gauss 変換**)
3. `stack`: 同じ5分割で fold ごとに標準化 → ロジスティック回帰を学習 → 検証foldを予測、を繰り返して**メタレベルでも out-of-fold な**スコアを得る。最後に全行で refit してテスト予測を作る

**なぜそうするのか**:
- **rank-gauss**: メンバーごとに出力スケールが違う(確率、ロジット、順位)。順位に直してから正規分位点に写すと、どのメンバーも同じ形の分布になり、線形モデルの係数が「重み」として素直に比較できる。外れ値の影響も潰れる。
- **C=0.03 の強めのL2正則化**: メンバーが200近くあり、互いに強く相関している。正則化が弱いと係数が暴れて(多重共線性)、わずかなノイズで重みが入れ替わる。強めに縛ることで**滑らかな加重平均に近づける**。
- **nested評価**: スタックを学習に使った行で評価すると必ず高く出る。fold内で学習し fold外で予測する形にして初めて、**報告された数字が汎化性能の推定として意味を持つ**。原著者の一文が的確 — "A stack scored in-sample reads high and means nothing."

> **用語**: *nested cross-validation(入れ子交差検証)* = ベースモデルのOOFを作る分割と、メタモデルを評価する分割を一致させ、メタレベルでも未見データで測る手法。スタッキングでの自己欺瞞を防ぐ標準的な作法。

In [ ]:
def rank01(v):
    return (rankdata(v, method="average") - 0.5) / len(v)


def gauss(v):
    return norm.ppf(np.clip(rank01(v), 1e-7, 1 - 1e-7))


def stack(cols_oof, cols_test, C=0.03):
    X = np.column_stack([gauss(c) for c in cols_oof])
    Xt = np.column_stack([gauss(c) for c in cols_test])
    pred = np.zeros(n)
    for itr, iva in FOLDS:
        sc = StandardScaler().fit(X[itr])
        m = LogisticRegression(C=C, max_iter=5000, solver="lbfgs", tol=1e-5)
        m.fit(sc.transform(X[itr]), y[itr])
        pred[iva] = m.decision_function(sc.transform(X[iva]))
    auc = roc_auc_score(y, pred)
    sc = StandardScaler().fit(X)
    m = LogisticRegression(C=C, max_iter=5000, solver="lbfgs", tol=1e-5)
    m.fit(sc.transform(X), y)
    return auc, pred, m.decision_function(sc.transform(Xt))

### Result: three arms, and the middle one is the base for everything after

1. **The full pool as loaded.** Comparison only. It includes the level 2 members.
2. **Base models only.** This is the base for the rest of the notebook.
3. **Base models plus my 50.** This is what gets submitted.

Dropping the blends from the submission is not tidiness. This notebook argues that level 2
members do not belong in a final selection, and submitting a blend that contains nine of
them would contradict that. It also makes the member set here the same one the leaderboard
table below was measured on, so those scores describe this pipeline rather than a
neighbouring run of mine.

**A reproduction check first, because everything after it depends on this.** adarsh1077's
notebook publishes a nested score of **0.970094** for his 177 member pool, and prints
**0.970083** for the same pool with the blends excluded. This notebook is an independent
reimplementation written from reading his loader, not a copy of his code, so those two
figures are the test of whether it is really his pipeline or merely something similar.

If the first arm below lands on 0.97009 and the second on 0.97008, the reimplementation is
faithful and the rest of the notebook is measuring what it claims to measure. If it does
not, treat every number after it as mine rather than his.

### 3つの腕(arm)で比較する — 再現チェックを最初に置く

**何をしているか**: (1) プール全体(ブレンド込み)、(2) ベースモデルのみ、(3) ベース+自作50本、の3通りでスタックを回してスコアを並べる。

**なぜそうするのか**: 特筆すべきは **腕1が「比較専用」である**こと。原著者は adarsh1077 の公開値(177メンバーで 0.970094、ブレンド除外で 0.970083)を**再現できるかどうかを最初の関門**にしている。彼は他人のコードをコピーせず読んで書き直しているので、この2つの数字が合わなければ「同じパイプラインを実装できていない」ことになり、**以降のすべての結論が無効**になる。

**再現チェックを先に置き、それに通らなければ先に進まない** — 実験設計として非常に強い型。腕2以降が主張の土台になるので、土台を先に検証している。

In [ ]:
# Arm 1, comparison only: the full pool as loaded, blends included. This is the arm that
# reproduces adarsh1077's published figure.
names_full = sorted(POOL_O)
auc_full, _, _ = stack([POOL_O[k] for k in names_full], [POOL_T[k] for k in names_full])
print(f"full pool as loaded {len(names_full):4d} members   nested OOF {auc_full:.7f}"
      f"   (includes {len(L2)} level 2 members)")

# Arm 2, THE BASE for everything after this: base models only.
base_cols_o = [POOL_O[k] for k in BASE_ONLY]
base_cols_t = [POOL_T[k] for k in BASE_ONLY]
auc_base, oof_base, test_base = stack(base_cols_o, base_cols_t)
print(f"base models only    {len(BASE_ONLY):4d} members   nested OOF {auc_base:.7f}"
      f"   ({auc_base - auc_full:+.7f} vs the full pool)")

# Arm 3, the submission: base models plus my 50.
plus_o = base_cols_o + [MINE_O[:, j] for j in range(MINE_O.shape[1])]
plus_t = base_cols_t + [MINE_T[:, j] for j in range(MINE_T.shape[1])]
auc_plus, oof_plus, test_plus = stack(plus_o, plus_t)
print(f"base + my 50        {len(plus_o):4d} members   nested OOF {auc_plus:.7f}"
      f"   ({auc_plus - auc_base:+.7f} from 50 arrays weaker than anything in the pool)")

# adarsh1077's published figures, for comparison. His "blends excluded" run drops the
# nine naji blends but KEEPS pub_ravi, so his 168 is our 167 plus that one array. The two
# are not the same member set and the numbers below should not be read as if they were.
print()
print(f"  adarsh1077 publishes 0.970094 for the full 177    -> here {auc_full:.7f}")
print(f"  and 0.970083 for 168, blends excluded but         -> here {auc_base:.7f} at "
      f"{len(BASE_ONLY)}")
print(f"  keeping pub_ravi, which this notebook also drops")
print()
print(f"  NOTE ON THE LEADERBOARD TABLE BELOW. Those scores were measured on 168 and 218")
print(f"  members. This notebook computes {len(BASE_ONLY)} and {len(plus_o)}: the same sets "
      f"minus pub_ravi,")
print(f"  the level 2 stack. One member of difference, measured at about -0.000001 in my")
print(f"  own audit, so the scores are comparable. They are not the identical set.")

## Why weak arrays still move a strong pool

The obvious reading of the result above is "more members is better". I do not think that is
what is happening, and the composition of the two sets says why.

The obvious reading is "more members is better". I do not think that is what is happening.

The public pool is almost entirely **discriminative**, and overwhelmingly boosted trees. The
three largest families across the libraries I hold are LightGBM, CatBoost and XGBoost, all
of them fitting `P(y | x)` directly.

**Almost none of my 50 is a gradient boosted tree.** Most of them are **generative**: they
fit the distribution of the features given the label and invert it, instead of fitting a
decision boundary at all. That is a different class of model, not a weaker version of the
same one.

A generative model of this data is not competitive on its own, and the solo AUCs above show
that plainly. But it makes **different mistakes**, because it is answering a different
question, and a stack is paid for uncorrelated error rather than for accuracy. That is the
whole of my explanation, and the correlation check below is the part of it you can verify
from this notebook.

I am deliberately not publishing what these models are or how they were built while the
competition is running. What I will say is the part that makes the result interpretable
rather than magical: they are not more trees.

### なぜ「弱い配列」が強いプールを動かすのか — 相関で検証する

**何をしているか**: 自作50本と、プールの合意(ベース168本のランク平均)との**スピアマン順位相関の中央値**を計算し、プールのメンバー同士の相関と比較する。

**なぜそうするのか**: 「メンバーを増やせばスコアが上がる」は表面的な読みで、原著者はそれを否定する。公開プールはほぼ全部が**識別モデル**(LightGBM/CatBoost/XGBoost = `P(y|x)` を直接学習)である一方、自作50本の多くは**生成モデル**(特徴量の分布 `P(x|y)` を学習して反転する)。

**アンサンブルが報酬を受け取るのは「精度」ではなく「誤りの非相関性」**。生成モデルは単体では弱いが、**違う質問に答えているので違う間違いをする**。だから加えると効く。この主張を「弱く、かつ合意から遠い」という2軸の数字で**検証可能な形にしている**のが本セルの価値。

> **用語**: *識別モデル(discriminative)* = 決定境界 `P(y|x)` を直接学習。*生成モデル(generative)* = `P(x|y)` を学習してベイズ則で反転。後者は単体精度で劣ることが多いが、誤りの構造が全く異なる。

In [ ]:
# The claim above, checkable: how correlated are my arrays with the pool, and with the
# pool's own consensus, compared to how correlated pool members are with each other?
from scipy.stats import spearmanr

# base_cols_o is the 168 base models, so the consensus below excludes the level 2
# members. Including a blend of the pool in the pool's own consensus would be circular.
pool_consensus = np.mean([rank01(c) for c in base_cols_o], axis=0)
mine_to_cons = [spearmanr(MINE_O[:, j], pool_consensus).statistic
                for j in range(MINE_O.shape[1])]
pool_to_cons = [spearmanr(c, pool_consensus).statistic for c in base_cols_o]
print(f"pool member vs pool consensus, median rank corr : {np.median(pool_to_cons):.4f}")
print(f"my array   vs pool consensus, median rank corr : {np.median(mine_to_cons):.4f}")
print(f"\nmy arrays are weaker AND further from the consensus, which is the combination a "
      f"stack can use.")

## Provenance: what I can verify and what I cannot

This is the section I would want to read in someone else's notebook.

The pool above is assembled from public datasets by several authors. They are not equally
verifiable, and the differences are not about quality. They are about whether I can
establish, from published evidence, that an array was produced under the same partition
this stack assumes.

The tiers I use:

| tier | what it means |
|---|---|
| mine | I trained it. |
| verified | The author published the training code, or a fold id file I could check directly, and it matches. |
| author stated | The author states the split in a published notebook and independent evidence is consistent with it, but the array builder itself was not published. |
| unverified | I could not establish the split from published material. |
| known different | Published evidence shows a different protocol. |

**The bottom two tiers, plainly, and the first one is mine to own.**

`pub_evg` uses `StratifiedKFold(n_splits=10)`. That is not an inference from a statistic, it
is what the published training code says. I did not train it. I adopted it into my own OOF
library early on, without checking, and my library description then told people that every
model in it used the 5-fold split with seed 42. That was not true, and correcting it here is
overdue.

Then the level 2 members, and there are more of them than I first wrote. `pub_ravi` is a
stack rather than a base model, and so are najiama's `naji_blend*` arrays, which are fitted
over this same pool. That makes part of what they contribute a re-weighting of members
sitting beside them in the same design.

This is not a footnote. **The pool loads 177 members, and only 168 of them are base
models.** The nine blends are among the strongest looking arrays in it, which is exactly
what you would expect from something fitted on the pool it is being scored inside, and it
is why a solo AUC leaderboard over these members is misleading. The cell below names them
and the results section measures what they are worth.

**A second case, and this one the author states himself.** boltuzamaki describes his
library as trained under five OR ten fold out-of-fold protocols, and reports that moving one
XGBoost from five folds to ten was worth about 0.00018 to him. That is his own published
account, not an inference of mine, and it means part of that library is on a different
partition from the one this stack assumes. Which part is not stated, and the filenames only
identify some of it.

The useful thing is that both halves of his own comparison are in this pool, so his figure
can be checked here rather than quoted. The cell below does that.

And a set of arrays in the pool come from a library whose training code was never published.
There is no statistic that settles a fold scheme, so for those the honest answer is simply
that I do not know, and no amount of blending experience changes that.

I am not presenting any of this as harmless, and I am not presenting the pool as unusable
either. For a final submission I would want the pool restricted to single model out-of-fold
arrays whose split I can establish. That is a different set from the one that scores
highest, which is the whole tension this notebook is about.

### プロヴェナンス(出所)の監査 — 分割スキームが違うと何が起きるか

**何をしているか**: 「別の分割で作られたと分かっているメンバー」(`pub_evg` は 10-fold、`pub_ravi` はレベル2スタック)をプールから探して solo AUC とともに印字する。さらに、同じ手法を5foldと10foldで作った2本(`foldsafe_te_xgb` と `foldsafe_te_xgb_10f`)が両方プールに入っているので、**fold数を増やすことの効果を引用ではなく実測で示す**。

**なぜそうするのか**: ここが本notebookの中核。10-fold で作られた OOF 配列は、自分の検証行を予測したモデルが**訓練データの約90%を既に見ている**(5-foldなら80%)。だから **OOF が実力より良く見え、メタモデルが過大な重みを与える**。

重要なのは、原著者が「これはリークではない」と正しく区別している点 — テスト予測自体は正常(テストにラベルは無いのでリークしようがない)。壊れるのは**重み付けの妥当性**だけ。だから「LBは上がるのに、CVとLBの乖離だけが静かに広がる」という、最も気づきにくい形で害が出る。

数字を「引用」せず**手元のプールで計算し直す**姿勢も一貫している。

In [ ]:
# The counts here are computed from the loaded pool, not typed in.
KNOWN_DIFFERENT = {"pub_evg": "StratifiedKFold(n_splits=10), read in the source",
                   "pub_ravi": "level 2 stack, not a base model"}
present = {k: v for k, v in KNOWN_DIFFERENT.items() if k in POOL_O}
print("members in this pool that I know are built differently:")
for k, v in present.items():
    print(f"  {k:10s} solo AUC {roc_auc_score(y, POOL_O[k]):.6f}   {v}")
if not present:
    print("  none found under those names in the attached datasets")

# boltuzamaki reports that moving one XGBoost from five folds to ten was worth about
# 0.00018 to him. Both streams are in this pool under names that state their fold count,
# so the figure is checkable here rather than quoted.
print()
_p = {k.split("/")[-1]: k for k in POOL_O}
_a, _b = _p.get("foldsafe_te_xgb_10f"), _p.get("foldsafe_te_xgb")
if _a and _b:
    _x, _y = roc_auc_score(y, POOL_O[_a]), roc_auc_score(y, POOL_O[_b])
    print(f"  foldsafe_te_xgb_10f  {_x:.6f}")
    print(f"  foldsafe_te_xgb      {_y:.6f}")
    print(f"  difference           {_x - _y:+.6f}   against the ~0.00018 he reports")
    print("  A 10-fold array is not wrong. Its OOF rows were predicted by models that had")
    print("  seen 90 percent of each validation fold, so it reads better here than it")
    print("  should, and the meta model weights it accordingly.")
else:
    print("  the two foldsafe_te_xgb streams were not found under those names in this pool")

## What a different fold scheme actually costs

This is the part I got wrong for a while, so it is worth stating carefully.

A member built under a different partition is **not** a leak into the leaderboard. Its test
predictions are honest, because there are no test labels to leak. The damage is confined to
its out-of-fold array:

For a row in my validation fold, a member built under a foreign 10-fold split was predicted
by a model that had already seen about 90 percent of that fold. So its OOF looks better than
it is, and the meta model gives it **too much weight**.

The consequence is a blend that is weighted wrongly, not a score that is invalid. That is
why adding such members can still raise the leaderboard: a strong model, over weighted, can
still help. It also means the harm does not show up where you would look for it. It shows up
as a **drift in the gap between cross validation and leaderboard**, which is the next
section.

I would rather hold a member like this than pretend it does not exist, and I would rather
not stake a final submission on a blend whose weights I know are distorted.

## Cross validation against the leaderboard, and where it stops helping

Three scores from my own submissions, measured on **the same member sets the notebook
computes above**: 168 base models, and 168 plus my 50, and 168 plus my whole unpublished
library.

**The middle row is close to checkable and the other two are not.** Nobody can see my
submission history, so the first and third rows you have to take as reported.

The middle row is one member away from what this notebook submits. Those scores were measured
on 168 pool members and on 168 plus my 50. This notebook drops one more array than that run
did, `pub_ravi`, the level 2 stack, so it computes 167 and 217. I would rather carry that
difference in the open than quietly keep a stack I have just argued against. Its measured
contribution to this pool was about -0.000001, so the scores are comparable, and if you fork
this notebook and submit its output you should land near 0.97119 rather than exactly on it.

| what | members | nested OOF | public LB | LB minus OOF |
|---|---|---|---|---|
| the pool alone | 168 | 0.9700832 | 0.97111 | +0.00103 |
| pool plus my worst 50 | 218 | 0.9701244 | 0.97119 | +0.00107 |
| pool plus all of my unpublished library | 597 | 0.9702399 | 0.97130 | +0.00106 |

Two things I take from this.

The gap between OOF and LB is stable at about +0.001 across all three, which is what you
want to see. When that gap **drops** while OOF rises, the blend has gained something the
leaderboard does not agree with. I use that as a warning sign. It needs no assumption about
how large a gain ought to be, which is why I find it more usable than comparing deltas, but
it is a heuristic I settled on from my own runs and not something this notebook establishes.

And the differences between these rows are small. The step from 168 members to 218 is
0.00008 of leaderboard. Repeated submissions of near identical blends move by a comparable
amount for no reason at all, so I would not read the ordering of these three as settled. I
report them because the direction is consistent, not because each step is individually
resolvable.

### The rungs in between

Measured in my own runs, not reproduced in this notebook. Same pipeline, same pool, adding
my arrays worst first:

| added | members | nested OOF | change |
|---|---|---|---|
| none | 168 | 0.9700832 | |
| worst 10 | 178 | 0.9700713 | -0.0000120 |
| worst 20 | 188 | 0.9700710 | -0.0000122 |
| worst 30 | 198 | 0.9700785 | -0.0000047 |
| worst 50 | 218 | 0.9701244 | +0.0000412 |
| worst 100 | 268 | 0.9701496 | +0.0000664 |
| worst 200 | 368 | 0.9701624 | +0.0000792 |
| all 429 | 597 | 0.9702399 | +0.0001567 |

**The first three rungs are negative.** Ten, twenty and thirty of my weakest arrays make the
stack worse under this pipeline, and it only turns positive somewhere between thirty and
fifty. So "add weak diverse members" is not free, and the count at which it starts paying is
not something I could have guessed. I published 50 rather than 30 because 30 does not work.

### The same ladder on a pool I can verify

The rungs above use the pool as it is. I also ran the same ladder on a **cut** version of it,
dropping every array whose fold scheme I could not establish from published material, plus
the level 2 stacks and the 10-fold array discussed above. That leaves 115 of the pool's
members. Measured in my own runs, same pipeline:

| added | members | nested OOF | change |
|---|---|---|---|
| none, verified pool only | 115 | 0.9698843 | |
| worst 50 | 165 | 0.9699300 | +0.0000457 |
| worst 100 | 215 | 0.9699612 | +0.0000770 |
| worst 200 | 315 | 0.9699835 | +0.0000992 |
| all 429 | 544 | 0.9700903 | +0.0002060 |

Two things I did not expect.

**My arrays are worth more to the verified pool than to the full one.** All of them together
add +0.0002060 here against +0.0001567 to the full pool. A thinner pool has more room.

**And the verified pool cannot catch up.** 115 verified members plus all 429 of mine reaches
0.9700903, which is roughly where the full pool starts before I add anything. Dropping the
unverifiable arrays costs about as much as my entire unpublished library is worth. That is
the trade, stated as a number, and it is why I am not going to pretend the clean version is
also the high scoring one.

There is one more arm I wanted and do not have: the same 544 member set under a wider meta
design. It ran out of memory twice and I have not rerun it.

### A related result that does not hold at this size

Blending my weakest arrays **with each other**, without the pool:

| arrays | blended OOF |
|---|---|
| worst 50 | 0.967173 |
| worst 100 | 0.968169 |
| worst 200 | 0.968959 |

The strongest single model I have trained scores 0.968734. So at 200 arrays the weakest
models in my library blend to more than my strongest single model, and at 50 and 100 they
do not. That comparison is between two numbers from my own library, so unlike the pipeline
results above you cannot check it from this notebook. I am reporting it.

I mention it because it is the more striking version of the same idea, and because it is
**not** what this notebook shows. The 50 arrays published here do not reach that result on
their own. Their value here is what they add to a pool that is already strong.

## Why 0.97130 is not my final submission

This is the argument promised at the top. The 0.97130 next to my name on the public
leaderboard is this pool plus my whole unpublished library, and I do not intend to select
it. Three reasons, in order of how much weight I put on them.

1. **Weights I know are distorted.** Not every member here is verifiable. Some come from a
   library whose training code was never published, so the split cannot be checked at all.
   At least one uses 10 folds, and it reached this pool through my own library. Some are
   level 2 stacks rather than base models, which makes them partly a re-weighting of members
   sitting beside them. For a final submission I would want the pool to be single model
   out-of-fold arrays only. The blend is optimised
   over out-of-fold values that are too good for those members.
2. **The public leaderboard cannot rank the top of this competition.** The cluster near the
   top sits inside the range that repeated near identical submissions move by anyway.
3. **Public and private have come apart before in this series.** In finished Season 6
   episodes the leading group on the public board has repeatedly been reordered on the
   private one, while whole board correlation stayed high. georgymamarin worked through this
   across seven boards in
   [Three of seven S6 boards erased the public top ten](https://www.kaggle.com/code/georgymamarin/three-of-seven-s6-boards-erased-the-public-top-ten),
   and it is worth reading before anyone selects on public rank, mine included.

So I am selecting on cross validation and on the OOF to leaderboard gap, using a member set
whose provenance I can state. This notebook is the other half of that decision: here is the
higher score, here is exactly what is in it, and here is why I am leaving it on the table.

## Submission

The blend of the pool plus my 50, ranked to `[0, 1]`. AUC only reads order, so the rank
transform changes nothing about the score and keeps the output in a familiar range.

### 提出ファイルの生成

**何をしているか**: ベース+自作50本のスタック出力を `rank01` で [0,1] に写し、`submission.csv` を書き出す。長さと有限性を `assert` で確認している。

**なぜそうするのか**: AUC は順序しか見ないので、ランク変換はスコアを変えない。それでも変換するのは、出力を見慣れた範囲に収めて**目視で異常に気づきやすくする**ため。`assert` で行数と NaN/inf を潰すのは、提出が形式エラーで弾かれる最頻の原因を機械的に排除する定石。

**最後に残る問い**: このnotebookが提出するのは 0.97119 だが、著者の公開LB最高は 0.97130。それでも後者を最終選択しないと明言している。理由は3つ — (1) 検証不能なメンバーによって重みが歪んでいる、(2) 公開LBの分解能が上位クラスタを識別できない、(3) このシリーズでは過去にpublic/privateが乖離している。**「一番高いスコア」と「一番信頼できるスコア」は別物**、というのが持ち帰るべき教訓。

In [ ]:
sub = pd.DataFrame({"id": te["id"].values, "addicted_label": rank01(test_plus)})
assert len(sub) == n_te and np.isfinite(sub["addicted_label"]).all()
sub.to_csv("submission.csv", index=False)
print(f"wrote submission.csv  {sub.shape}")
print(f"base models only  {len(BASE_ONLY)} members  nested OOF {auc_base:.7f}")
print(f"base + my 50      {len(plus_o)} members  nested OOF {auc_plus:.7f}   <- submitted")
sub.head()